# 03_v4 - SAM2 Tracking Refinement

Obiettivi:

1. partire dal tracking RF-DETR + SAM2 V1;
2. validare le track SAM2 tramite RF-DETR;
3. utilizzare matching one-to-one per individuare track duplicate/stale;
4. rimuovere false track persistenti;
5. analizzare la frammentazione degli ID;
6. individuare possibili track da ricongiungere;
7. produrre una nuova versione del tracking da valutare con TrackEval.

La Ground Truth non viene utilizzata per modificare le predizioni.

In [1]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from scipy.optimize import linear_sum_assignment


DRIVE_ROOT = Path(
    "/Users/marcodalbis/Library/CloudStorage/"
    "GoogleDrive-marco.dalbis@gmail.com/"
    "Il mio Drive/"
    "basketball-thesis"
)

SEQUENCE_NAME = "v_00HRwkvvjtQ_c001"

FRAMES_DIR = (
    DRIVE_ROOT
    / "datasets"
    / "sportsmot"
    / "val"
    / SEQUENCE_NAME
    / "img1"
)

SAM2_V1_CSV = (
    DRIVE_ROOT
    / "experiments"
    / "sam2_tracking"
    / f"{SEQUENCE_NAME}_tracks.csv"
)

print("Frames:", FRAMES_DIR)
print("Frames exists:", FRAMES_DIR.exists())

print()

print("SAM2 V1:", SAM2_V1_CSV)
print("SAM2 V1 exists:", SAM2_V1_CSV.exists())

Frames: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/datasets/sportsmot/val/v_00HRwkvvjtQ_c001/img1
Frames exists: True

SAM2 V1: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/sam2_tracking/v_00HRwkvvjtQ_c001_tracks.csv
SAM2 V1 exists: True


In [2]:
sam2_df = pd.read_csv(
    SAM2_V1_CSV
)

sam2_df = (
    sam2_df
    .sort_values(
        ["frame", "track_id"]
    )
    .reset_index(drop=True)
)

print(
    "Observations:",
    len(sam2_df)
)

print(
    "Unique IDs:",
    sam2_df["track_id"].nunique()
)

print(
    "Frame range:",
    sam2_df["frame"].min(),
    "->",
    sam2_df["frame"].max()
)

print(
    "Duplicate frame/ID:",
    sam2_df.duplicated(
        ["frame", "track_id"]
    ).sum()
)

Observations: 12334
Unique IDs: 20
Frame range: 1 -> 1162
Duplicate frame/ID: 0


In [3]:
def bbox_iou(box_a, box_b):

    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)

    inter = iw * ih

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union = area_a + area_b - inter

    if union <= 0:
        return 0.0

    return inter / union

In [4]:
from rfdetr import RFDETRNano


CHECKPOINT_PATH = (
    DRIVE_ROOT
    / "checkpoints"
    / "rfdetr_nano_10epochs"
    / "checkpoint_best_total.pth"
)

model = RFDETRNano(
    pretrain_weights=str(
        CHECKPOINT_PATH
    )
)

print("RF-DETR loaded")

/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[2026-10-08 01:28:09] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-10-08 01:28:09] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-10-08 01:28:09] [WARNING] rf-detr - Checkpoint has 1 classes but model is configured for 90. Using checkpoint class count (1). Pass num_classes=1 to suppress this warning.


RF-DETR loaded


In [5]:
def detect_players_rfdetr(
    frame_number,
    threshold=0.30
):

    frame_path = (
        FRAMES_DIR
        / f"{frame_number:06d}.jpg"
    )

    frame_bgr = cv2.imread(
        str(frame_path)
    )

    if frame_bgr is None:

        raise FileNotFoundError(
            frame_path
        )

    frame_rgb = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2RGB
    )

    detections = model.predict(
        frame_rgb,
        threshold=threshold
    )

    boxes = [
        np.asarray(
            box,
            dtype=np.float32
        )
        for box in detections.xyxy
    ]

    confidences = [
        float(conf)
        for conf in detections.confidence
    ]

    return boxes, confidences

In [6]:
test_boxes, test_conf = (
    detect_players_rfdetr(
        1,
        threshold=0.30
    )
)

print(
    "Frame 1 detections:",
    len(test_boxes)
)

[2026-10-08 01:28:09] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.inference(dtype=torch.float16).


Frame 1 detections: 8


In [7]:
def bbox_iou(box_a, box_b):

    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(
        0.0,
        ix2 - ix1
    )

    ih = max(
        0.0,
        iy2 - iy1
    )

    inter = iw * ih

    area_a = (
        max(0.0, ax2 - ax1)
        *
        max(0.0, ay2 - ay1)
    )

    area_b = (
        max(0.0, bx2 - bx1)
        *
        max(0.0, by2 - by1)
    )

    union = (
        area_a
        + area_b
        - inter
    )

    if union <= 0:
        return 0.0

    return inter / union

In [8]:
rfdetr_cache = {}


for frame_number in tqdm(
    range(1, 1163),
    desc="RF-DETR detections"
):

    boxes, confidences = (
        detect_players_rfdetr(
            frame_number,
            threshold=0.30
        )
    )

    rfdetr_cache[
        frame_number
    ] = {
        "boxes": [
            np.asarray(
                box,
                dtype=np.float32
            )
            for box in boxes
        ],
        "confidences": confidences,
    }


print(
    "Cached frames:",
    len(rfdetr_cache)
)

RF-DETR detections:   0%|          | 0/1162 [00:00<?, ?it/s]

Cached frames: 1162


## Matching ONE-TO-ONE

In [9]:
sam2_df[
    "detector_iou_1to1"
] = 0.0

sam2_df[
    "detector_supported_1to1"
] = False

sam2_df[
    "matched_detection"
] = -1


IOU_SUPPORT_THRESHOLD = 0.15


for frame_number, frame_tracks in tqdm(
    sam2_df.groupby("frame"),
    total=sam2_df["frame"].nunique(),
    desc="One-to-one matching"
):

    row_indices = (
        frame_tracks.index.to_list()
    )

    sam2_boxes = (
        frame_tracks[
            ["x1", "y1", "x2", "y2"]
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    detector_boxes = (
        rfdetr_cache[
            int(frame_number)
        ]["boxes"]
    )

    if (
        len(sam2_boxes) == 0
        or
        len(detector_boxes) == 0
    ):
        continue

    iou_matrix = np.zeros(
        (
            len(sam2_boxes),
            len(detector_boxes)
        ),
        dtype=np.float32
    )

    for i, sam_box in enumerate(
        sam2_boxes
    ):

        for j, det_box in enumerate(
            detector_boxes
        ):

            iou_matrix[i, j] = (
                bbox_iou(
                    sam_box,
                    det_box
                )
            )

    matched_rows, matched_cols = (
        linear_sum_assignment(
            -iou_matrix
        )
    )

    for row_idx, det_idx in zip(
        matched_rows,
        matched_cols
    ):

        iou = float(
            iou_matrix[
                row_idx,
                det_idx
            ]
        )

        dataframe_index = (
            row_indices[
                row_idx
            ]
        )

        sam2_df.loc[
            dataframe_index,
            "detector_iou_1to1"
        ] = iou

        if (
            iou
            >= IOU_SUPPORT_THRESHOLD
        ):

            sam2_df.loc[
                dataframe_index,
                "detector_supported_1to1"
            ] = True

            sam2_df.loc[
                dataframe_index,
                "matched_detection"
            ] = int(det_idx)

One-to-one matching:   0%|          | 0/1162 [00:00<?, ?it/s]

In [12]:
support_1to1 = (
    sam2_df
    .groupby("track_id")
    .agg(
        observations=(
            "frame",
            "size"
        ),

        start_frame=(
            "frame",
            "min"
        ),

        end_frame=(
            "frame",
            "max"
        ),

        supported=(
            "detector_supported_1to1",
            "sum"
        ),

        mean_iou=(
            "detector_iou_1to1",
            "mean"
        ),
    )
)


support_1to1[
    "support_ratio"
] = (
    support_1to1["supported"]
    /
    support_1to1["observations"]
)


support_1to1 = (
    support_1to1
    .sort_values(
        "support_ratio"
    )
)


display(
    support_1to1
)

,observations,start_frame,end_frame,supported,mean_iou,support_ratio
track_id,,,,,,
9,440,101,540,24,0.038323,0.054545
16,90,821,910,7,0.059307,0.077778
6,540,1,540,96,0.091743,0.177778
5,540,1,540,162,0.177254,0.300000
1,540,1,540,251,0.311504,0.464815
11,450,191,640,388,0.634667,0.862222
14,882,281,1162,826,0.726501,0.936508
2,81,1,81,77,0.522098,0.950617
4,1090,1,1090,1046,0.763265,0.959633


In [13]:
TRACK_SUPPORT_THRESHOLD = 0.50

MIN_TRACK_LENGTH = 30


stale_track_ids = (
    support_1to1[
        (
            support_1to1[
                "support_ratio"
            ]
            < TRACK_SUPPORT_THRESHOLD
        )
        &
        (
            support_1to1[
                "observations"
            ]
            >= MIN_TRACK_LENGTH
        )
    ]
    .index
    .astype(int)
    .tolist()
)


print(
    "Automatically rejected IDs:",
    stale_track_ids
)

print(
    "Number:",
    len(stale_track_ids)
)

Automatically rejected IDs: [9, 16, 6, 5, 1]
Number: 5


In [14]:
clean_df = (
    sam2_df[
        ~sam2_df[
            "track_id"
        ].isin(
            stale_track_ids
        )
    ]
    .copy()
)


print(
    "Original observations:",
    len(sam2_df)
)

print(
    "Clean observations:",
    len(clean_df)
)

print(
    "Removed:",
    len(sam2_df)
    - len(clean_df)
)

print()

print(
    "Original IDs:",
    sam2_df[
        "track_id"
    ].nunique()
)

print(
    "Clean IDs:",
    clean_df[
        "track_id"
    ].nunique()
)

Original observations: 12334
Clean observations: 10184
Removed: 2150

Original IDs: 20
Clean IDs: 15


In [15]:
tracks_per_frame = (
    clean_df
    .groupby("frame")
    ["track_id"]
    .nunique()
)


print(
    "Mean tracks/frame:",
    round(
        tracks_per_frame.mean(),
        2
    )
)

print(
    "Min:",
    tracks_per_frame.min()
)

print(
    "Max:",
    tracks_per_frame.max()
)

print()

print(
    "Frames > 10:",
    int(
        (
            tracks_per_frame > 10
        ).sum()
    )
)

print(
    "Frames = 10:",
    int(
        (
            tracks_per_frame == 10
        ).sum()
    )
)

print(
    "Frames < 10:",
    int(
        (
            tracks_per_frame < 10
        ).sum()
    )
)

Mean tracks/frame: 8.76
Min: 4
Max: 10

Frames > 10: 0
Frames = 10: 605
Frames < 10: 557


In [24]:
# ============================================================
# SAVE SAM2 V2 - 18 IDs
# ============================================================

SAM2_V2_DIR = (
    DRIVE_ROOT
    / "experiments"
    / "sam2_tracking_v2"
)

SAM2_V2_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SAM2_V2_CSV = (
    SAM2_V2_DIR
    / f"{SEQUENCE_NAME}_tracks.csv"
)

columns_to_save = [
    "frame",
    "track_id",
    "x1",
    "y1",
    "x2",
    "y2",
    "confidence",
    "class_id",
]

clean_df[
    columns_to_save
].to_csv(
    SAM2_V2_CSV,
    index=False
)

print("Saved:", SAM2_V2_CSV)
print("Rows:", len(clean_df))
print("IDs:", clean_df["track_id"].nunique())
print("Exists:", SAM2_V2_CSV.exists())

Saved: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/sam2_tracking_v2/v_00HRwkvvjtQ_c001_tracks.csv
Rows: 10184
IDs: 15
Exists: True


In [26]:
original_ids = set(
    sam2_df["track_id"].unique()
)

clean_ids = set(
    clean_df["track_id"].unique()
)

removed_ids = sorted(
    original_ids - clean_ids
)

remaining_ids = sorted(
    clean_ids
)

print(
    "Original IDs:",
    sorted(original_ids)
)

print()

print(
    "Remaining IDs:",
    remaining_ids
)

print()

print(
    "Removed IDs:",
    removed_ids
)

print()

print(
    "Original count:",
    len(original_ids)
)

print(
    "Clean count:",
    len(clean_ids)
)

Original IDs: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]

Remaining IDs: [np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(7), np.int64(8), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(17), np.int64(18), np.int64(19)]

Removed IDs: [np.int64(1), np.int64(5), np.int64(6), np.int64(9), np.int64(16)]

Original count: 20
Clean count: 15


In [27]:
display(
    support_1to1.loc[
        support_1to1.index.isin(
            removed_ids
        )
    ]
)

,observations,start_frame,end_frame,supported,mean_iou,support_ratio
track_id,,,,,,
9,440,101,540,24,0.038323,0.054545
16,90,821,910,7,0.059307,0.077778
6,540,1,540,96,0.091743,0.177778
5,540,1,540,162,0.177254,0.300000
1,540,1,540,251,0.311504,0.464815


In [25]:
# ============================================================
# CREATE VIDEO - SAM2 V2
# ============================================================

VIDEO_OUTPUT = (
    SAM2_V2_DIR
    / f"{SEQUENCE_NAME}_tracking_v2.mp4"
)

first_frame = cv2.imread(
    str(
        FRAMES_DIR
        / "000001.jpg"
    )
)

if first_frame is None:
    raise FileNotFoundError(
        FRAMES_DIR / "000001.jpg"
    )

height, width = first_frame.shape[:2]

FPS = 25

writer = cv2.VideoWriter(
    str(VIDEO_OUTPUT),
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    (width, height)
)

for frame_number in tqdm(
    range(1, 1163),
    desc="Creating tracking video"
):

    frame_path = (
        FRAMES_DIR
        / f"{frame_number:06d}.jpg"
    )

    frame = cv2.imread(
        str(frame_path)
    )

    if frame is None:
        continue

    frame_tracks = (
        clean_df[
            clean_df["frame"]
            == frame_number
        ]
    )

    for _, row in frame_tracks.iterrows():

        x1 = int(row["x1"])
        y1 = int(row["y1"])
        x2 = int(row["x2"])
        y2 = int(row["y2"])

        track_id = int(
            row["track_id"]
        )

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"ID {track_id}",
            (
                x1,
                max(20, y1 - 8)
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

    writer.write(frame)

writer.release()

print("Video saved:")
print(VIDEO_OUTPUT)
print("Exists:", VIDEO_OUTPUT.exists())

Creating tracking video:   0%|          | 0/1162 [00:00<?, ?it/s]

Video saved:
/Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/sam2_tracking_v2/v_00HRwkvvjtQ_c001_tracking_v2.mp4
Exists: True


In [28]:
# ============================================================
# CREATE VIDEO - ORIGINAL SAM2 V1 (20 IDs)
# ============================================================

VIDEO_OUTPUT_V1 = (
    DRIVE_ROOT
    / "experiments"
    / "sam2_tracking"
    / f"{SEQUENCE_NAME}_tracking_v1_20ids.mp4"
)

VIDEO_OUTPUT_V1.parent.mkdir(
    parents=True,
    exist_ok=True
)

first_frame = cv2.imread(
    str(
        FRAMES_DIR
        / "000001.jpg"
    )
)

height, width = first_frame.shape[:2]

writer = cv2.VideoWriter(
    str(VIDEO_OUTPUT_V1),
    cv2.VideoWriter_fourcc(*"mp4v"),
    25,
    (width, height)
)

for frame_number in tqdm(
    range(1, 1163),
    desc="Creating SAM2 V1 video"
):

    frame_path = (
        FRAMES_DIR
        / f"{frame_number:06d}.jpg"
    )

    frame = cv2.imread(
        str(frame_path)
    )

    if frame is None:
        continue

    frame_tracks = (
        sam2_df[
            sam2_df["frame"]
            == frame_number
        ]
    )

    for _, row in frame_tracks.iterrows():

        x1 = int(row["x1"])
        y1 = int(row["y1"])
        x2 = int(row["x2"])
        y2 = int(row["y2"])

        track_id = int(
            row["track_id"]
        )

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"ID {track_id}",
            (
                x1,
                max(20, y1 - 8)
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

    writer.write(frame)

writer.release()

print("Saved:")
print(VIDEO_OUTPUT_V1)

print(
    "IDs:",
    sam2_df["track_id"].nunique()
)

Creating SAM2 V1 video:   0%|          | 0/1162 [00:00<?, ?it/s]

Saved:
/Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/sam2_tracking/v_00HRwkvvjtQ_c001_tracking_v1_20ids.mp4
IDs: 20
